# Phase 7: Probabilistic Demand Forecasting & Uncertainty Quantification
## MarketPulse: E-Commerce Demand Forecasting Engine

Point forecasts are fundamentally inadequate for inventory planning. Supply chain decisions—such as setting **Safety Stock ($SS$)**, calculating **Reorder Points ($ROP$)**, and managing **Stockout Risk**—require full probability distributions or calibrated prediction intervals.

### Core Implementations in Phase 7:
1. **Multi-Quantile LightGBM**: Direct quantile estimation across $\alpha \in \{0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95\}$.
2. **Monotonic Rearrangement**: Guarantees zero quantile crossings (Chernozhukov et al. 2010).
3. **Distribution-Free Conformal Prediction**: Finite-sample coverage guarantees ($P(y \in [l, u]) \ge 1 - \beta$).
4. **Zero-Inflated Intermittent Forecasting**: Exact quantile derivation for intermittent SKUs ($Q(\alpha) = 0$ when $\alpha \le 1 - \pi$).
5. **Proper Scoring Rules**: Pinball Loss, CRPS, PICP, MPIW, and Winkler Score.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecasting.probabilistic import (
    QuantileLightGBMForecaster,
    ConformalIntervalForecaster,
    IntermittentProbabilisticForecaster,
    DEFAULT_QUANTILES,
)
from src.evaluation.metrics import (
    pinball_loss,
    mean_pinball_loss,
    prediction_interval_coverage_probability,
    mean_prediction_interval_width,
    winkler_score,
    continuous_ranked_probability_score,
)
from src.hierarchy.hierarchy_builder import HierarchyTree
from src.hierarchy.aggregation import HierarchicalAggregator

## 1. Load Processed Time-Series Data

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'processed' / 'm5_processed.parquet'
df = pd.read_parquet(data_path)
tree = HierarchyTree(metadata_df=df)
aggregator = HierarchicalAggregator(hierarchy=tree.m5_hierarchy)
bottom_wide = aggregator.prepare_bottom_wide_dataframe(df)
all_wide = aggregator.aggregate_bottom_to_all_levels(bottom_wide)

# Set daily frequency
all_wide.index = pd.date_range(start=all_wide.index[0], periods=len(all_wide), freq='D')
bottom_wide.index = pd.date_range(start=bottom_wide.index[0], periods=len(bottom_wide), freq='D')

horizon = 28
train_size = len(all_wide) - horizon
y_train_tot = all_wide['Total'].iloc[:train_size]
y_test_tot = all_wide['Total'].iloc[train_size:]

print(f"Training window: {len(y_train_tot)} days, Test horizon: {len(y_test_tot)} days.")

## 2. Multi-Quantile Gradient Boosting (LightGBM Fan Chart)

In [ ]:
lgb_q = QuantileLightGBMForecaster(
    quantiles=DEFAULT_QUANTILES,
    lags=[1, 7, 14],
    rolling_windows=[7, 14],
    n_estimators=60,
    random_seed=42,
)
lgb_q.fit(y_train_tot)
quantiles_df = lgb_q.predict_quantiles(horizon=horizon)

# Verify zero quantile crossing
crossings = (np.diff(quantiles_df.values, axis=1) < -1e-6).sum()
print(f"Quantile crossings detected: {crossings} (Guaranteed 0 by Monotonic Rearrangement)")
quantiles_df.head()

### Fan Chart Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6), dpi=150)
dates = y_test_tot.index

# Plot historical tail for context
ax.plot(y_train_tot.index[-21:], y_train_tot.values[-21:], color='dimgray', label='History (Last 3 Wks)', lw=1.5)
ax.plot(dates, y_test_tot.values, color='black', lw=2.2, label='Actual Demand (Ground Truth)')

# Fan bands
ax.fill_between(dates, quantiles_df[0.05], quantiles_df[0.95], color='royalblue', alpha=0.15, label='90% Interval [q0.05, q0.95]')
ax.fill_between(dates, quantiles_df[0.10], quantiles_df[0.90], color='royalblue', alpha=0.25, label='80% Interval [q0.10, q0.90]')
ax.fill_between(dates, quantiles_df[0.25], quantiles_df[0.75], color='royalblue', alpha=0.40, label='50% Interval (IQR) [q0.25, q0.75]')
ax.plot(dates, quantiles_df[0.50], color='navy', lw=2.0, linestyle='--', label='Median Forecast (q0.50)')

ax.set_title('MarketPulse Multi-Quantile Probabilistic Demand Forecast (Total Business Demand)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales Units')
ax.legend(loc='upper left', frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Conformal Prediction Intervals (Finite-Sample Distribution-Free Coverage)

In [ ]:
from src.forecasting.baselines import create_seasonal_naive

conformal = ConformalIntervalForecaster(
    forecaster=create_seasonal_naive(sp=7).estimator,
    nominal_coverage=0.90,
)
conformal.fit(y_train_tot, val_fraction=0.25)
pt_conf, low_conf, up_conf = conformal.predict_intervals(horizon=horizon)

picp_90 = prediction_interval_coverage_probability(y_test_tot, low_conf, up_conf)
mpiw_90 = mean_prediction_interval_width(low_conf, up_conf)
winkler_90 = winkler_score(y_test_tot, low_conf, up_conf, nominal_coverage=0.90)

print("Conformal Prediction Interval Metrics (Nominal = 90%):")
print(f" - Empirical Coverage (PICP):  {picp_90:.1f}%")
print(f" - Mean Interval Width (MPIW): {mpiw_90:.2f} units")
print(f" - Winkler Interval Score:     {winkler_90:.2f}")

## 4. Zero-Inflated Probabilistic Modeling for Intermittent SKUs

In [ ]:
# Select a sparse intermittent SKU
zero_fracs = (bottom_wide.iloc[:train_size] == 0).mean()
intermittent_sku = zero_fracs.idxmax()
y_sku_train = bottom_wide[intermittent_sku].iloc[:train_size]
y_sku_test = bottom_wide[intermittent_sku].iloc[train_size:]

inter_model = IntermittentProbabilisticForecaster()
inter_model.fit(y_sku_train)
sku_quantiles = inter_model.predict_quantiles(horizon=horizon, quantiles=DEFAULT_QUANTILES)

print(f"SKU '{intermittent_sku}' Demand Characteristics:")
print(f" - Historical Zero Proportion: {(y_sku_train == 0).mean() * 100:.1f}%")
print(f" - Fitted Demand Occurrence Probability (pi): {inter_model.occurrence_prob_:.3f}")
print(f" - Conditional Non-Zero Demand Mean: {inter_model.size_mean_:.2f} units")
print("Derived Quantiles:")
sku_quantiles.iloc[[0]]

## 5. Quantitative Scoring Rules & Probabilistic Leaderboard

In [ ]:
crps_val = continuous_ranked_probability_score(y_test_tot, quantiles_df)
pinball_val = mean_pinball_loss(y_test_tot, quantiles_df)

print("Proper Scoring Rule Evaluation (Total Corporate Demand):")
print(f" - Continuous Ranked Probability Score (CRPS): {crps_val:.4f}")
print(f" - Mean Pinball Loss (Quantile Loss):           {pinball_val:.4f}")